In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# トランスパイラの最適化レベルを設定する

<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降の使用を推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
実際の量子デバイスはノイズやゲートエラーの影響を受けるため、回路の深さとゲート数を削減するように最適化することで、それらの回路の実行から得られる結果を大幅に改善できます。
[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 関数には、必須の位置引数 `optimization_level` があり、トランスパイラが回路の最適化にどれだけの労力をかけるかを制御します。この引数には、0、1、2、3 のいずれかの整数を指定できます。
最適化レベルが高いほど、コンパイル時間が長くなる代わりに、より最適化された回路が生成されます。
次の表は、各設定で実行される最適化の内容を説明しています。

<table>
  <thead>
    <tr>
      <th>最適化レベル</th>
      <th>説明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        最適化なし：主にハードウェア特性評価に使用
        - 基本的な変換のみ
        - レイアウト/ルーティング：`TrivialLayout` を使用し、仮想量子ビットと同じ物理量子ビット番号を選択し、`SabreSwap` を使用して動作するように SWAP を挿入
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        軽度の最適化：
        -   レイアウト/ルーティング：まず `TrivialLayout` でレイアウトを試みます。追加の SWAP が必要な場合は、`SabreSwap` を使用して SWAP 数が最小となるレイアウトを見つけ、次に `VF2LayoutPostLayout` を使用してグラフ内の最適な量子ビットを選択します。
        -   `InverseCancellation`
        -   1Q ゲートの最適化
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        中程度の最適化：
          - レイアウト/ルーティング：最適化レベル 1（trivial なし）+ より大きな探索深度と最適化関数の試行回数によるヒューリスティック最適化。`TrivialLayout` を使用しないため、物理量子ビット番号と仮想量子ビット番号を同じにしようとする試みは行われません。
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        高度な最適化：
        - 最適化レベル 2 + より大きな労力・試行回数でレイアウト/ルーティングをさらにヒューリスティック最適化
        - [Cartan の KAK 分解](https://arxiv.org/abs/quant-ph/0507171) を使用した 2 量子ビットブロックの再合成
        - ユニタリ性を破るパス：
          * `OptimizeSwapBeforeMeasure`：SWAP を回避するために測定をリアレンジ
          * `RemoveDiagonalGatesBeforeMeasure`：測定結果に影響しない測定前のゲートを削除
      </td>
    </tr>
  </tbody>
</table>

## 最適化レベルの動作
2 量子ビットゲートは通常、最も大きなエラーの原因であるため、結果として得られる回路の 2 量子ビットゲート数を数えることで、トランスパイルの「ハードウェア効率」をおおよそ定量化できます。
ここでは、ランダムユニタリに SWAP ゲートが続く入力回路に対して、異なる最適化レベルを試してみます。

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

例では `FakeSherbrooke` モックバックエンドを使用します。まず、最適化レベル 0 でトランスパイルしてみましょう。

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

トランスパイルされた回路には、2 量子ビット ECR ゲートが 6 つあります。

最適化レベル 1 で繰り返してみます：

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

トランスパイルされた回路にはまだ 6 つの ECR ゲートがありますが、1 量子ビットゲートの数が減少しています。

最適化レベル 2 で繰り返してみます：

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

最適化レベル 1 と同じ結果が得られます。最適化レベルを上げても、必ずしも違いが生まれるわけではないことに注意してください。

最適化レベル 3 で再度繰り返してみます：

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

今度は ECR ゲートが 3 つだけになりました。これは、最適化レベル 3 では Qiskit が 2 量子ビットゲートのブロックを再合成しようとするためで、どんな 2 量子ビットゲートも最大 3 つの ECR ゲートで実装できます。`approximation_degree` を 1 未満の値に設定すると、さらに ECR ゲートを減らすことができます。これにより、トランスパイラはゲート分解に若干の誤差をもたらす可能性のある近似を行うことが許可されます（[トランスパイルのよく使われるパラメーター](common-parameters#approximation-degree)を参照）：

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

この回路は ECR ゲートが 2 つだけですが、近似回路です。この回路の効果が正確な回路とどのように異なるかを理解するために、この回路が実装するユニタリ演算子と正確なユニタリとの間の忠実度（フィデリティ）を計算できます。計算を実行する前に、まず 127 量子ビットを含むトランスパイルされた回路を、アクティブな量子ビット（2 つ）のみを含む回路に縮小します。

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>